In [1]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
from pyrosm import OSM
import matplotlib.pyplot as plt
import osmnx as ox

study_area_buffered_gdf = gpd.read_file(r"output\cycling_network.gpkg", layer="study_area")
study_area_buffered = study_area_buffered_gdf.geometry.union_all()

In [ ]:
# G = ox.graph_from_polygon(study_area_buffered, network_type="all")
# nodes, edges = ox.graph_to_gdfs(G)

In [ ]:
# nodes.to_file("output/all_network.gpkg", layer='nodes_espoo', driver="GPKG")
# edges.to_file("output/all_network.gpkg", layer='edges_espoo', driver="GPKG")

In [2]:
nodes = gpd.read_file("output/all_network.gpkg", layer='nodes_espoo')
edges = gpd.read_file("output/all_network.gpkg", layer='edges_espoo')

In [3]:
nodes.head()

,osmid,y,x,street_count,highway,ref,junction,railway,geometry
0,130190,60.187995,24.828789,3,NaN,NaN,NaN,NaN,POINT (24.82879 60.188)
1,131941,60.181524,24.849017,3,NaN,NaN,NaN,NaN,POINT (24.84902 60.18152)
2,134888,60.180122,24.837602,3,NaN,NaN,NaN,NaN,POINT (24.8376 60.18012)
3,318650,60.222705,24.662233,3,NaN,NaN,NaN,NaN,POINT (24.66223 60.22271)
4,318651,60.224979,24.675076,3,motorway_junction,30B,NaN,NaN,POINT (24.67508 60.22498)


In [4]:
edges.head()

,u,v,key,osmid,highway,name,oneway,reversed,length,lanes,...,bridge,ref,width,service,access,junction,tunnel,est_width,area,geometry
0,130190,299447488,0,7212304,service,Rakentajanaukio,True,False,5.453623,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LINESTRING (24.82879 60.188, 24.82888 60.18797)"
1,130190,25030457,0,76732306,tertiary,Otakaari,False,False,20.170718,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LINESTRING (24.82879 60.188, 24.82898 60.18815)"
2,130190,291818800,0,76732306,tertiary,Otakaari,False,True,46.181942,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LINESTRING (24.82879 60.188, 24.82835 60.18764)"
3,131941,269328887,0,"[264473404, 179043485, 24787814]",secondary,Kuusisaarentie,True,False,64.316253,"['1', '2']",...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LINESTRING (24.84902 60.18152, 24.84943 60.181..."
4,131941,922854269,0,"[81178546, 78722622]",secondary,Kuusisaarentie,False,True,253.186317,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"LINESTRING (24.84902 60.18152, 24.84631 60.181..."


In [5]:
# Intersections
intersections = nodes[nodes["street_count"] >= 3].copy()

In [6]:
intersections.head()

,osmid,y,x,street_count,highway,ref,junction,railway,geometry
0,130190,60.187995,24.828789,3,NaN,NaN,NaN,NaN,POINT (24.82879 60.188)
1,131941,60.181524,24.849017,3,NaN,NaN,NaN,NaN,POINT (24.84902 60.18152)
2,134888,60.180122,24.837602,3,NaN,NaN,NaN,NaN,POINT (24.8376 60.18012)
3,318650,60.222705,24.662233,3,NaN,NaN,NaN,NaN,POINT (24.66223 60.22271)
4,318651,60.224979,24.675076,3,motorway_junction,30B,NaN,NaN,POINT (24.67508 60.22498)


In [7]:
# Relevant tags for cycling crossings
tags = {
    "highway": ["traffic_signals", "crossing"],
    "crossing": True,
    "crossing:signals": True,
    "crossing:markings": True,
    "cycleway": "crossing",
    "bicycle": True,
    "traffic_calming": True,
}

features = ox.features_from_polygon(study_area_buffered, tags)

In [8]:
features.to_file("output/all_network.gpkg", layer='features_espoo', driver="GPKG")

In [ ]:
features.explore()

In [9]:
# Keep point features
point_features = features[features.geometry.geom_type.isin(["Point", "MultiPoint"])].copy()

# Project for spatial join
target_crs = "EPSG:3879"

intersections_proj = intersections.to_crs(target_crs)
point_features_proj = point_features.to_crs(target_crs)

intersection_info = gpd.sjoin_nearest(
    point_features_proj,
    intersections_proj[["street_count", "geometry"]],
    how="left",
    distance_col="dist_to_intersection"
)

intersection_info = intersection_info[intersection_info["dist_to_intersection"] <= 20]

In [10]:
intersection_info.head()

geometry          highway  ref  \
element id                                                                 
node    693946    POINT (25475104.336 6674287.918)         crossing  NaN   
        694161    POINT (25486224.395 6673028.091)  traffic_signals  NaN   
        19475524  POINT (25491070.427 6675229.815)         crossing  NaN   
        25030586   POINT (25490802.99 6675152.686)         crossing  NaN   
        25031758  POINT (25490833.753 6674322.282)         crossing  NaN   

                 destination name name:fi name:sv      crossing  \
element id                                                        
node    693946           NaN  NaN     NaN     NaN  uncontrolled   
        694161           NaN  NaN     NaN     NaN           NaN   
        19475524         NaN  NaN     NaN     NaN        marked   
        25030586         NaN  NaN     NaN     NaN  uncontrolled   
        25031758         NaN  NaN     NaN     NaN           NaN   

                 crossing:island   kerb  ... charging_station:output  \
element id                               ...                           
node    693946                no  flush  ...                     NaN   
        694161               NaN    NaN  ...                     NaN   
        19475524              no    NaN  ...                     NaN   
        25030586             NaN    NaN  ...                     NaN   
        25031758              no  flush  ...                     NaN   

                 payment:credit_cards payment:debit_cards  \
element id                                                  
node    693946                    NaN                 NaN   
        694161                    NaN                 NaN   
        19475524                  NaN                 NaN   
        25030586                  NaN                 NaN   
        25031758                  NaN                 NaN   

                 socket:type2_combo:output surveillance surveillance:type  \
element id                                                                  
node    693946                         NaN          NaN               NaN   
        694161                         NaN          NaN               NaN   
        19475524                       NaN          NaN               NaN   
        25030586                       NaN          NaN               NaN   
        25031758                       NaN          NaN               NaN   

                 parking:both:maxstay:conditional index_right street_count  \
element id                                                                   
node    693946                                NaN          28            4   
        694161                                NaN       14031            4   
        19475524                              NaN          72            4   
        25030586                              NaN          76            4   
        25031758                              NaN       80144            3   

                 dist_to_intersection  
element id                             
node    693946               0.000000  
        694161              10.060553  
        19475524             0.000000  
        25030586             0.000000  
        25031758            10.316131  

[5 rows x 528 columns]

In [11]:
cols = [
    "highway",
    "crossing",
    "crossing:signals",
    "crossing:markings",
    "cycleway",
    "bicycle",
    "traffic_calming",
    "dist_to_intersection",
    "geometry",
]

available_cols = [c for c in cols if c in intersection_info.columns]
intersection_info = intersection_info[available_cols]
intersection_info.head()

highway      crossing crossing:signals  \
element id                                                         
node    693946           crossing  uncontrolled              NaN   
        694161    traffic_signals           NaN              NaN   
        19475524         crossing        marked              NaN   
        25030586         crossing  uncontrolled              NaN   
        25031758         crossing           NaN               no   

                 crossing:markings cycleway bicycle traffic_calming  \
element id                                                            
node    693946                 NaN      NaN     NaN             NaN   
        694161                 NaN      NaN     NaN             NaN   
        19475524             zebra      NaN     NaN             NaN   
        25030586               NaN      NaN     NaN             NaN   
        25031758                no      NaN     NaN             NaN   

                  dist_to_intersection                          geometry  
element id                                                                
node    693946                0.000000  POINT (25475104.336 6674287.918)  
        694161               10.060553  POINT (25486224.395 6673028.091)  
        19475524              0.000000  POINT (25491070.427 6675229.815)  
        25030586              0.000000   POINT (25490802.99 6675152.686)  
        25031758             10.316131  POINT (25490833.753 6674322.282)

In [16]:
ic = intersection_info.copy().reset_index()

ic["is_signalized"] = (
    (ic.get("highway") == "traffic_signals") |
    (ic.get("crossing") == "traffic_signals") |
    (ic.get("crossing:signals") == "yes")
)

ic["is_marked_crossing"] = ic.get("crossing").isin(["marked", "uncontrolled"])
ic["is_bike_crossing"] = ic.get("cycleway") == "crossing"
ic["has_traffic_calming"] = ic.get("traffic_calming").notna()

In [17]:
import numpy as np

ic["intersection_type"] = np.select(
    [
        ic.is_bike_crossing & ic.is_signalized,
        ic.is_bike_crossing,
        ic.is_signalized,
        ic.is_marked_crossing,
        ic.has_traffic_calming
    ],
    [
        "bike_signalized_crossing",
        "bike_crossing",
        "signalized_intersection",
        "marked_crossing",
        "traffic_calmed_intersection"
    ],
    default="uncontrolled_intersection"
)

ic.head()

,element,id,highway,crossing,crossing:signals,crossing:markings,cycleway,bicycle,traffic_calming,dist_to_intersection,geometry,is_signalized,is_marked_crossing,is_bike_crossing,has_traffic_calming,intersection_type
0,node,693946,crossing,uncontrolled,NaN,NaN,NaN,NaN,NaN,0.000000,POINT (25475104.336 6674287.918),False,True,False,False,marked_crossing
1,node,694161,traffic_signals,NaN,NaN,NaN,NaN,NaN,NaN,10.060553,POINT (25486224.395 6673028.091),True,False,False,False,signalized_intersection
2,node,19475524,crossing,marked,NaN,zebra,NaN,NaN,NaN,0.000000,POINT (25491070.427 6675229.815),False,True,False,False,marked_crossing
3,node,25030586,crossing,uncontrolled,NaN,NaN,NaN,NaN,NaN,0.000000,POINT (25490802.99 6675152.686),False,True,False,False,marked_crossing
4,node,25031758,crossing,NaN,no,no,NaN,NaN,NaN,10.316131,POINT (25490833.753 6674322.282),False,False,False,False,uncontrolled_intersection


In [ ]:
ic.explore(
    column="intersection_type",
    categorical=True,
    legend=True
)

8829

In [2]:
import geopandas as gpd
# ic.to_file("output/all_network.gpkg", layer='intersection_info', driver="GPKG")

ic = gpd.read_file("output/all_network.gpkg", layer='intersection_info')
